In [ ]:
########## This code is used to pull river discharge data from Geoglows for the reaches surrounding study area dams ##########

In [ ]:
## Import the packages needed
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import s3fs
import xarray
import datetime
from tqdm.notebook import tqdm

In [ ]:
### Find the River IDs we need to pull from GeoGlows ### 
# Pull in GeoGlows Lines By VPU -- Note: These files are big and need to be filtered 
# Location of Stream Files 
GeoGlows_path = r"F:\Insert_File_Path_of_GeoGlows_Centerlines\streams_"  # Update this file path
# List of units 
VPU_List = [702, 703, 704, 706, 709, 712, 713, 714, 715]

## Get the Streams files, filter, and combine them into one place
GeoGlows_BigRivers = gpd.GeoDataFrame()

for i in tqdm(VPU_List): 
    filename  = GeoGlows_path + str(i)+ '.gpkg' 
    GeoGlows_Streams = gpd.read_file(filename)
    GeoGlows_St_Sub = GeoGlows_Streams[GeoGlows_Streams['strmOrder']>= 4]
    Output = pd.concat([GeoGlows_BigRivers,GeoGlows_St_Sub])
    GeoGlows_BigRivers = Output[:]

GeoGlows_BigRivers

In [ ]:
### All Potential Pull in Profiles of Interest ###
Potential_Profiles = pd.read_csv(r"F:\Insert_File_Path_Here\Usable_Profiles_List.csv") # Update this file path

## Get Dams of Interest ## 
Dams_List = Potential_Profiles['Assgn_dam'].unique().tolist()

# Pull in all Study Dams
GrodDams = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\Study_Dams.shp", columns=['grod_id', 'type', 'lon', 'lat']) # Update this file path

# Filter the dams
Selected_Dams = GrodDams[GrodDams["grod_id"].isin(Dams_List)]

# Preview
Selected_Dams

In [ ]:
#### Find the River Segments within a 2 km buffer of the Dam ####
# Dam's nearest GeoGlows segments
GeoGlows_byDam = pd.DataFrame()

## Loop through dams and select the segments
for i in tqdm(Dams_List):
    # Select dam and make sure it's projection is correct
    Grodsub = GrodDams[GrodDams['grod_id'] == i]
    Grodsub_proj = Grodsub.to_crs(GeoGlows_BigRivers.crs)
    point_center = Grodsub_proj.geometry.iloc[0] ## The location the buffer is centered on
    buffer_distance_meters = 2000 # Distance in meters

    # Create the buffer
    buffer_polygon = point_center.buffer(buffer_distance_meters)

    # Select lines within the buffer
    lines_in_buffer = GeoGlows_BigRivers[GeoGlows_BigRivers.intersects(buffer_polygon)]

    # Save the links to a DF
    LineInfo = lines_in_buffer[['LINKNO', 'DSLINKNO']]
    LineInfo['Assgn_dam'] = i
    Output = pd.concat([LineInfo, GeoGlows_byDam])
    GeoGlows_byDam = Output[:]

GeoGlows_byDam = GeoGlows_byDam.reset_index(drop=True)

# Preview
GeoGlows_byDam

In [ ]:
#### Set up the Function to Pull the GeoGlows Data ####
def pull_daily_Q(reach_id, start_date, end_date):
    start_date = datetime.datetime.strptime(start_date, "%Y-%m-%d").date()
    end_date = datetime.datetime.strptime(end_date, "%Y-%m-%d").date()
    sim_begin = datetime.date(1940, 1, 1)
    first_index = start_date - sim_begin
    second_index = end_date - sim_begin
    bucket_uri = "s3://geoglows-v2/retrospective/daily.zarr"
    region_name = "us-west-2"
    s3 = s3fs.S3FileSystem(anon=True, client_kwargs=dict(region_name=region_name))
    s3store = s3fs.S3Map(root=bucket_uri, s3=s3, check=False)
    ds = xarray.open_zarr(s3store)
    df = (
        ds["Q"]
        .sel(river_id=reach_id, time=slice(start_date, end_date))
        .to_dataframe()
    )

    df = df.reset_index().set_index("time").pivot(columns="river_id", values="Q")
    return df

In [ ]:
## Pull the GeoGlows Data ##
# Get a list of Reaches to pull 
GeoGlows_ID_List = GeoGlows_byDam['LINKNO'].unique().tolist()

# Set up parameters for pull
Start_Date  = '2013-03-01'
End_Date = '2024-12-31'

## Set up dataframe
GeoGlows_Q = pd.DataFrame()

## Loop through and Pull the Discharge
for i in tqdm(GeoGlows_ID_List):
    # Pull the data
    Reach_Q = pull_daily_Q(i, Start_Date, End_Date)
    # Add supplemental information
    Reach_Q['ReachID'] = i 
    Reach_Q = Reach_Q.rename(columns={i: 'Discharge'}) 
    Reach_Q['DailyQDiff'] = Reach_Q['Discharge'].diff()
    # Join info all together
    Output = pd.concat([Reach_Q,GeoGlows_Q])
    GeoGlows_Q = Output[:]

GeoGlows_Q = GeoGlows_Q.reset_index(drop = False)

# Preview
GeoGlows_Q

In [ ]:
### Save the GeoGlows Data
GeoGlows_Q.to_csv(r"F:\Insert_File_Output_Path_Here\QbyReach_All.csv") # Update this file path

In [ ]:
################ Prepare the Data for Joining to Other ML Tables ################

In [ ]:
## We want a copy of profiles with a datetime colum
Profiles_Copy = Potential_Profiles[:]
Profiles_Copy["Date"] = pd.to_datetime(Profiles_Copy[['Year', 'Month', 'Day']])
Profiles_Copy = Profiles_Copy[['Assgn_dam', 'Date']]
Profiles_Copy

In [ ]:
## Join the GeoGlows ReachIDs to the profiles
Profiles_GeoGlowsID = pd.merge(left = Profiles_Copy, right = GeoGlows_byDam, on = 'Assgn_dam', how = 'outer' )

## Join the IDs to their Q data
QbyDam_All = pd.merge(Profiles_GeoGlowsID, GeoGlows_Q, left_on = ['Date', 'LINKNO'], right_on = ['time', 'ReachID' ], how = 'left' )

In [ ]:
## Try Getting the Average Q
QbyDam_Avg = QbyDam_All.groupby(['Assgn_dam', 'Date']).agg({"Discharge":'mean'})
QbyDam_Avg.columns = ["Avg_Q"]
QbyDam_Avg = QbyDam_Avg.reset_index()
QbyDam_Avg

In [ ]:
## Export the file
QbyDam_Avg.to_csv(r"F:\Insert_File_Output_Path_Here\Avg_Q_by_Dam.csv") # Update this file path